# Digit Recognizer - Kaggle Competition

This notebook implements a solution for the [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) competition using PyTorch. 

**Objective**: Classify handwritten digits (0-9) from grayscale images of 28x28 pixels.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

# Check versions and hardware acceleration status
print(f"PyTorch version: {torch.__version__}")
print(f"MKL Active (Intel Optimization): {torch.backends.mkl.is_available()}")

## 1. Load Data

The data is provided in CSV format where each row (after the header) is one image. 
- `train.csv`: Includes a 'label' column and 784 pixel columns (28x28).
- `test.csv`: Only includes the 784 pixel columns.

In [ ]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Train dataset: {train_df.shape[0]} images")
print(f"Test dataset: {test_df.shape[0]} images")

## 2. Exploratory Data Analysis (EDA)

Visualizing the data helps us understand the input format. Pixels are values from 0 (white) to 255 (black).

In [ ]:
# Extract labels (y) and features (X)
y = train_df['label'].values
X = train_df.drop('label', axis=1).values

# Plot the first 10 digits to verify data quality
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    # Reshape the 784 flat vector back to 28x28 grayscale image
    plt.imshow(X[i].reshape(28, 28), cmap='gray')
    plt.title(f"Digit: {y[i]}")
    plt.axis('off')
plt.show()

## 3. Data Preprocessing

Deep learning models perform better when inputs are scaled and structured correctly.

In [ ]:
# 1. Normalization: Scale pixels to [0, 1] range
X = X / 255.0

# 2. Split: 90% Training, 10% Validation to monitor overfitting
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# 3. PyTorch Dataset Wrapper
class MNISTDataset(Dataset):
    def __init__(self, X, y=None):
        # Reshape to (Batch, Channels, Height, Width) -> (N, 1, 28, 28)
        self.X = torch.tensor(X, dtype=torch.float32).reshape(-1, 1, 28, 28)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# 4. DataLoaders: Efficiently feed data in batches of 64
train_loader = DataLoader(MNISTDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(MNISTDataset(X_val, y_val), batch_size=64, shuffle=False)

## 4. CNN Model Architecture

Convolutional Neural Networks are the gold standard for image recognition. 
- **Conv2d**: Finds local patterns (lines, curves).
- **MaxPool2d**: Reduces image size to focus on important features.
- **Dropout**: Randomly ignores neurons during training to prevent "memorization" (overfitting).

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self):
        super(DigitCNN, self).__init__()
        # Feature Extraction Layers
        self.conv_layers = nn.Sequential(
            # Input: 1 channel (gray). Output: 32 feature maps.
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # Size becomes 14x14
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Size becomes 7x7
        )
        # Classification Layers
        self.fc_layers = nn.Sequential(
            # Flattened input: 64 maps * 7 width * 7 height
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2), # 20% dropout rate
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # Flatten before dense layers
        x = self.fc_layers(x)
        return x

model = DigitCNN()
criterion = nn.CrossEntropyLoss()           # Standard for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adaptive learning rate optimizer

## 5. Training and Evaluation

We loop through the data multiple times (**epochs**). Each time, the model adjusts its weights to reduce the error (**loss**).

In [ ]:
epochs = 5
for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()      # Clear old gradients
        outputs = model(images)    # Forward pass: predictable
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()            # Backward pass: calculate adjustment
        optimizer.step()           # Update weights
        running_loss += loss.item()
    
    # --- Validation Phase ---
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad(): # Disable gradient calc to save memory/time
        for images, labels in val_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_loss:.4f} | Val Accuracy: {accuracy:.2f}%")

## 6. Submission Generation

Finally, we predict the labels for the 28,000 images in the test set.

In [ ]:
# Prepare test data (normalize only, no labels)
X_test_scaled = test_df.values / 255.0
test_dataset = MNISTDataset(X_test_scaled)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model.eval()
predictions = []
with torch.no_grad():
    for images in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        predictions.extend(predicted.numpy())

# Format for Kaggle: ImageId (1 to 28000), Label
submission = pd.DataFrame({
    'ImageId': np.arange(1, len(predictions) + 1),
    'Label': predictions
})
submission.to_csv('submission.csv', index=False)
print("Success! submission.csv has been saved.")